# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Duchalsoham12/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [16]:
# Capstone question check

print("Research question:")
print("Can observed content and performance signals help identify declining content?")

print("\nDecision supported:")
print("Prioritize content for human review and possible refresh.")

Research question:
Can observed content and performance signals help identify declining content?

Decision supported:
Prioritize content for human review and possible refresh.


In [17]:
import os
import pandas as pd
import numpy as np

# GitHub repository
repo_path = "flyrank-ml-internship"

# Clone repository if needed
if not os.path.exists(repo_path):
    !git clone https://github.com/Duchalsoham12/flyrank-ml-internship.git

# Find dataset automatically
csv_path = None

for root, dirs, files in os.walk(repo_path):
    for file in files:
        if file == "content_refresh_anonymized.csv":
            csv_path = os.path.join(root, file)
            break
    if csv_path:
        break

if csv_path is None:
    raise FileNotFoundError(
        "content_refresh_anonymized.csv was not found."
    )

# Load dataset
df = pd.read_csv(csv_path)

# Create target
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Dataset loaded successfully")
print("Path:", csv_path)
print("Shape:", df.shape)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

display(df.head())

Dataset loaded successfully
Path: flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Shape: (30000, 45)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


## 1. Question

### Research question

Can observed content and performance signals help identify content that may be declining?

### Decision supported

The analysis is designed to help a content team prioritize pages for human review and possible refresh. The model provides directional decision-support rather than an automatic decision about what content should be changed.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [18]:
# Data summary

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nClients:", df["client_id"].nunique())

print("\nContent types:")
print(df["content_type"].value_counts().head())

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

Rows: 30000
Columns: 45

Clients: 32

Content types:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 2. Data

The analysis uses the anonymized content-performance dataset provided for the project. The dataset contains content attributes, search and performance signals, engagement measures, freshness information, and trend fields.

The analysis excludes direct identifiers such as client_id and content_id from model features. Trend fields used to construct the outcome label are also excluded from the final feature set to reduce direct target leakage.

The analysis uses the available historical windows in the dataset rather than introducing external private data.

In [19]:
print("Rows:", len(df))
...

Rows: 30000


In [20]:
# CAPSTONE MODEL SETUP
# Run this cell before Section 3

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# 1. Grouped 80/20 split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

# 2. Select model features
excluded = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

features = [
    c for c in df.columns
    if c not in excluded
]

X_train = train[features]
y_train = train["is_declining_label"]

X_test = test[features]
y_test = test["is_declining_label"]

# 3. Identify feature types
numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

# 4. Preprocessing
preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numeric_features
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

# 5. Random Forest
model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

# 6. Train
model.fit(X_train, y_train)

# 7. Predictions
pred_prob = model.predict_proba(X_test)[:, 1]

# 8. Precision@50
top_n = min(50, len(y_test))
top_indices = np.argsort(pred_prob)[::-1][:top_n]

honest_p50 = y_test.iloc[top_indices].mean()

# 9. Checks
print("Model setup complete.")
print("Features:", len(features))
print("Train rows:", len(train))
print("Test rows:", len(test))
print(
    "Client overlap:",
    len(set(train["client_id"]) & set(test["client_id"]))
)
print("Precision@50:", round(honest_p50, 3))

Model setup complete.
Features: 40
Train rows: 23837
Test rows: 6163
Client overlap: 0
Precision@50: 1.0


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [21]:
print("Model: Random Forest")
print("Validation: 80/20 grouped by client")
print("Final feature count:", len(features))

print("\nExcluded fields:")
print([
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
])

print("\nClient overlap:",
      len(set(train["client_id"]) & set(test["client_id"])))

print("Precision@50:",
      round(honest_p50, 3))

Model: Random Forest
Validation: 80/20 grouped by client
Final feature count: 40

Excluded fields:
['content_id', 'client_id', 'trend_direction', 'trend_pct', 'is_declining_label']

Client overlap: 0
Precision@50: 1.0


## 3. Methodology

The target is a binary declining-content label. A record is labeled declining when its observed trend_direction is "down".

A Random Forest classifier is used because it can handle numeric and categorical features and non-linear relationships.

The final feature set uses content, search, engagement, freshness, and traffic signals. content_id, client_id, trend_direction, trend_pct, and the target label are excluded to reduce direct leakage.

Validation uses an 80/20 grouped-by-client split so that the same client does not appear in both training and testing. The leakage audit checks that target-defining fields are not included as model features.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [22]:
# Results vs baseline

# IMPORTANT:
# Replace 0.240 with your actual Week-5 baseline
baseline_p50 = 0.240

results = pd.DataFrame({
    "method": [
        "Week-5 baseline",
        "Random Forest"
    ],
    "precision_at_50": [
        baseline_p50,
        honest_p50
    ]
})

results["difference_vs_baseline"] = (
    results["precision_at_50"] - baseline_p50
)

display(results)

print(
    "Observed Random Forest Precision@50:",
    round(honest_p50, 3)
)

,method,precision_at_50,difference_vs_baseline
0,Week-5 baseline,0.24,0.00
1,Random Forest,1.00,0.76


Observed Random Forest Precision@50: 1.0


## 4. Results (vs baseline)

The Random Forest is compared with the Week-5 baseline using Precision@50 on the same held-out, client-grouped test set. The difference is treated as a measured result under this validation design, not as a guarantee of future performance.

## 5. Limitations

*What this work cannot claim.*

In [23]:
print("Random Forest Precision@50:",
      round(honest_p50, 3))

print("Week-5 baseline:",
      baseline_p50)

print("Difference:",
      round(honest_p50 - baseline_p50, 3))

print("\nClient overlap:",
      len(set(train["client_id"]) & set(test["client_id"])))

Random Forest Precision@50: 1.0
Week-5 baseline: 0.24
Difference: 0.76

Client overlap: 0


## 5. Limitations

The Random Forest achieved a measured Precision@50 of 1.00 on the held-out client-grouped test set, compared with the Week-5 baseline of 0.24. This is an observed result for this dataset and validation split, not proof that the model will achieve the same performance on future data.

The result should be treated as directional decision-support. It cannot establish that the model causes content improvement, and it should not be used as an automatic decision system. Human review is required before taking action on any ranked recommendation.

The unusually high measured score should also be investigated for possible leakage, feature overlap, or differences between the label construction and the prediction features.

In [24]:
top50 = test.iloc[np.argsort(pred_prob)[::-1][:50]].copy()

print("Top 50 actual declining labels:")
print(top50["is_declining_label"].value_counts())

display(
    top50[
        [
            "content_id",
            "is_declining_label",
            "trend_direction",
            "trend_pct",
            "content_age_days",
            "days_since_last_update"
        ]
    ].head(20)
)

Top 50 actual declining labels:
is_declining_label
1    50
Name: count, dtype: int64


,content_id,is_declining_label,trend_direction,trend_pct,content_age_days,days_since_last_update
25063,content_29884c0f9255,1,down,-78.0,223,102
3329,content_eb3b2c3bbc34,1,down,-96.2,117,20
2139,content_41538bdb1b1e,1,down,-98.6,97,8
23480,content_9234f5075e7a,1,down,-93.0,95,20
17707,content_f6bf66378677,1,down,-86.1,141,20
14343,content_9ac61c04930e,1,down,-53.9,275,104
29681,content_9e8671965fff,1,down,-82.4,95,20
13704,content_c6bb205d5263,1,down,-49.1,95,20
18789,content_8b08ec7fc725,1,down,-100.0,237,103
3340,content_e5fd30b6e33b,1,down,-68.0,141,20


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 6. Ranked recommendations

The recommendation queue ranks content by the model's measured decline score. Higher-scored content is prioritized for human review.

Reason codes provide simple supporting signals such as stale content, low engagement, or negative trend. These recommendations are directional decision-support and should not trigger automatic content changes.

In [25]:
# Create the ranked recommendation queue

queue = test[
    [
        "content_id",
        "content_age_days",
        "days_since_last_update",
        "engagement_rate",
        "ai_traffic_pct",
        "trend_direction",
        "trend_pct"
    ]
].copy()

# Model's estimated decline probability
queue["decline_score"] = pred_prob


# Generate reason codes
def get_reason(row):
    reasons = []

    if (
        pd.notna(row["days_since_last_update"])
        and row["days_since_last_update"] > 180
    ):
        reasons.append("STALE_CONTENT")

    if (
        pd.notna(row["engagement_rate"])
        and row["engagement_rate"] < 0.50
    ):
        reasons.append("LOW_ENGAGEMENT")

    if (
        pd.notna(row["trend_pct"])
        and row["trend_pct"] < 0
    ):
        reasons.append("NEGATIVE_TREND")

    if not reasons:
        reasons.append("MODEL_PRIORITY")

    return "|".join(reasons)


queue["reason_code"] = queue.apply(
    get_reason,
    axis=1
)

# Rank highest score first
queue = queue.sort_values(
    "decline_score",
    ascending=False
).reset_index(drop=True)

queue["priority_rank"] = np.arange(
    1,
    len(queue) + 1
)

print("Recommendation queue created.")
print("Total recommendations:", len(queue))

display(
    queue[
        [
            "priority_rank",
            "content_id",
            "decline_score",
            "reason_code"
        ]
    ].head(20)
)

Recommendation queue created.
Total recommendations: 6163


,priority_rank,content_id,decline_score,reason_code
0,1,content_29884c0f9255,0.976667,LOW_ENGAGEMENT|NEGATIVE_TREND
1,2,content_eb3b2c3bbc34,0.970000,LOW_ENGAGEMENT|NEGATIVE_TREND
2,3,content_41538bdb1b1e,0.966667,LOW_ENGAGEMENT|NEGATIVE_TREND
3,4,content_9234f5075e7a,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND
4,5,content_9ac61c04930e,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND
5,6,content_f6bf66378677,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND
6,7,content_9e8671965fff,0.956667,LOW_ENGAGEMENT|NEGATIVE_TREND
7,8,content_c6bb205d5263,0.956667,LOW_ENGAGEMENT|NEGATIVE_TREND
8,9,content_8b08ec7fc725,0.953333,LOW_ENGAGEMENT|NEGATIVE_TREND
9,10,content_e5fd30b6e33b,0.950000,LOW_ENGAGEMENT|NEGATIVE_TREND


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [26]:
# Create output folder

output_dir = "flyrank-ml-internship/work/outputs"
os.makedirs(output_dir, exist_ok=True)

# 1. Model results
results_path = os.path.join(
    output_dir,
    "capstone_results.csv"
)

results.to_csv(
    results_path,
    index=False
)

# 2. Ranked recommendations
recommendations_path = os.path.join(
    output_dir,
    "capstone_recommendations.csv"
)

queue.to_csv(
    recommendations_path,
    index=False
)

# 3. Monitoring summary
monitoring_summary = pd.DataFrame({
    "metric": [
        "Test rows",
        "Test clients",
        "Decline rate",
        "Precision@50",
        "Baseline Precision@50"
    ],
    "value": [
        len(test),
        test["client_id"].nunique(),
        round(y_test.mean(), 3),
        round(honest_p50, 3),
        baseline_p50
    ]
})

monitoring_path = os.path.join(
    output_dir,
    "capstone_monitoring.csv"
)

monitoring_summary.to_csv(
    monitoring_path,
    index=False
)

print("Artifacts exported successfully.")
print("\nFiles:")

for path in [
    results_path,
    recommendations_path,
    monitoring_path
]:
    print("-", path)

Artifacts exported successfully.

Files:
- flyrank-ml-internship/work/outputs/capstone_results.csv
- flyrank-ml-internship/work/outputs/capstone_recommendations.csv
- flyrank-ml-internship/work/outputs/capstone_monitoring.csv


In [27]:
print("CAPSTONE ARTIFACT CHECK")
print("=======================")

print("Results rows:", len(results))
print("Recommendation rows:", len(queue))
print("Monitoring rows:", len(monitoring_summary))

print("\nOutput directory:")

for file in os.listdir(output_dir):
    print("-", file)

CAPSTONE ARTIFACT CHECK
Results rows: 2
Recommendation rows: 6163
Monitoring rows: 5

Output directory:
- capstone_recommendations.csv
- capstone_results.csv
- capstone_monitoring.csv


In [28]:
print("ML-12 complete.")
print("Dataset rows:", len(df))
print("Clients:", df["client_id"].nunique())
print("Test rows:", len(test))
print("Precision@50:", round(honest_p50, 3))
print("Recommendation queue:", len(queue))
print("Artifacts exported:", 3)

ML-12 complete.
Dataset rows: 30000
Clients: 32
Test rows: 6163
Precision@50: 1.0
Recommendation queue: 6163
Artifacts exported: 3


## 7. Artifacts the paper embeds

The paper uses the measured results table, ranked recommendation queue, and monitoring summary as supporting artifacts.

These files provide reproducible evidence for the model comparison and the recommended content-review priorities. The outputs are intended for decision-support and should be interpreted together with the validation and limitation notes.

# ML-12 — Demo and Communication

## 5-minute Demo Outline

1. **Question — 30 seconds**
   - Identify content that may be declining.
   - Support human review and prioritization.

2. **Data — 45 seconds**
   - 30,000 anonymized content records.
   - 32 clients.
   - Content, search, engagement, freshness, and traffic signals.

3. **Method — 60 seconds**
   - Random Forest classifier.
   - Declining label based on observed downward trend.
   - 80/20 grouped-by-client validation.
   - Leakage-sensitive fields excluded.

4. **Results — 60 seconds**
   - Week-5 baseline Precision@50: 0.24.
   - Random Forest Precision@50: 1.00 on this held-out split.
   - This is an observed result for this dataset and validation design.

5. **Action Queue — 45 seconds**
   - 6,163 items ranked by measured decline score.
   - Reason codes explain prioritization.
   - Highest-ranked items receive human review first.

6. **Limitations — 30 seconds**
   - Results are directional decision-support.
   - No causal claims.
   - Human review is required before action.

## Employer-Facing Summary

I built and validated a machine-learning workflow to prioritize potentially declining content for human review using anonymized content-performance data. The workflow included feature preparation, a Random Forest model, client-grouped validation, leakage checks, model comparison, and a ranked action queue. The final outputs provide measured, directional decision-support rather than automated content decisions.

## Social-Post Cut

Built a content-signal modeling workflow that turns observed performance signals into a ranked review queue.

Using an anonymized dataset of 30,000 content records, I tested a Random Forest model with client-grouped validation and leakage checks.

The model achieved a measured Precision@50 of 1.00 on the held-out split, compared with a 0.24 baseline. The result is treated as directional decision-support rather than a guarantee of future performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
